<a href="https://colab.research.google.com/github/AnkaRezaa/machine_learning_254107023002/blob/main/jobsheet%20dataset%20AntiBurnOut/JS06-regresi/06_Regression_BurnoutScore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS06-regresi: Analisis Regresi Prediksi Skor Burnout - Dataset AntiBurnOut

**Identitas Mahasiswa:**
- **Nama**  : MUHAMMAD ANKA REZA
- **NIM**   : 254107023002
- **Kelas** : TI/3H

---

### Deskripsi Praktikum:
Praktikum ini menerapkan algoritma regresi (Linear Regression, Polynomial Regression, Support Vector Regression, dan Random Forest Regressor) untuk memprediksi nilai kontinu **`Burnout_Score` (skor 0 – 100)** berdasarkan parameter gaya hidup, jam kerja, dan kondisi psikologis responden. Evaluasi dilakukan menggunakan metrik $R^2$, MAE, dan RMSE.

### Langkah 1: Memuat Data dan Preprocessing Fitur

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

dataset_candidates = [
    'mental_health_burnout_prediction_dataset.csv',
    '../mental_health_burnout_prediction_dataset.csv',
    'jobsheet dataset AntiBurnOut/mental_health_burnout_prediction_dataset.csv'
]
dataset_path = next((p for p in dataset_candidates if os.path.exists(p)), 'mental_health_burnout_prediction_dataset.csv')
df = pd.read_csv(dataset_path)

# Fitur prediktor regresi dan target kontinu
num_features = ['Work_Hours_Per_Week', 'Sleep_Hours', 'Anxiety_Score', 
                'Depression_Score', 'Mood_Score', 'Emotional_Stability', 
                'Physical_Activity_Hours', 'Screen_Time_Hours']

X = df[num_features].copy()
for col in num_features:
    X[col] = X[col].fillna(X[col].median())

y = df['Burnout_Score'].values

# Split data 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Data Latih Regresi: {X_train_scaled.shape}, Data Uji: {X_test_scaled.shape}")

### Langkah 2: Pelatihan dan Evaluasi Model Regresi (Linear, Polynomial, Random Forest)

In [ ]:
# 1. Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

# 2. Polynomial Regression (Degree 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled[:, :4])  # subset 4 fitur utama agar komputasi ringan
X_test_poly = poly.transform(X_test_scaled[:, :4])
poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)
y_pred_poly = poly_model.predict(X_test_poly)

# 3. Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

# Tabel Perbandingan Evaluasi
results = {
    'Model': ['Linear Regression', 'Polynomial Regression (Deg 2)', 'Random Forest Regressor'],
    'R2 Score': [r2_score(y_test, y_pred_lr), r2_score(y_test, y_pred_poly), r2_score(y_test, y_pred_rf)],
    'MAE': [mean_absolute_error(y_test, y_pred_lr), mean_absolute_error(y_test, y_pred_poly), mean_absolute_error(y_test, y_pred_rf)],
    'RMSE': [np.sqrt(mean_squared_error(y_test, y_pred_lr)), np.sqrt(mean_squared_error(y_test, y_pred_poly)), np.sqrt(mean_squared_error(y_test, y_pred_rf))]
}

res_df = pd.DataFrame(results)
print("=== Perbandingan Metrik Evaluasi Regresi ===")
res_df

### Langkah 3: Visualisasi Prediksi vs Nilai Aktual

In [ ]:
plt.figure(figsize=(12, 5))

# Scatter Aktual vs Prediksi
plt.subplot(1, 2, 1)
plt.scatter(y_test[:200], y_pred_rf[:200], alpha=0.6, color='royalblue', edgecolors='k')
plt.plot([0, 100], [0, 100], 'r--', linewidth=2, label='Garis Ideal (y=x)')
plt.title('Random Forest: Aktual vs Prediksi (200 Sampel)', fontsize=12)
plt.xlabel('Nilai Aktual Burnout Score')
plt.ylabel('Nilai Prediksi')
plt.legend()
plt.grid(True)

# Residual Plot
plt.subplot(1, 2, 2)
residuals = y_test - y_pred_rf
sns.histplot(residuals, kde=True, color='crimson')
plt.title('Distribusi Residual (Error)', fontsize=12)
plt.xlabel('Residual (Aktual - Prediksi)')
plt.ylabel('Frekuensi')

plt.tight_layout()
plt.show()

### Kesimpulan JS06-regresi:
1. Random Forest Regressor menghasilkan koefisien determinasi ($R^2$) tertinggi dengan tingkat error terendah dibandingkan model linier biasa, membuktikan adanya hubungan non-linier antara variabel beban kerja/stres dengan skor keparahan burnout.
2. Distribusi residual berpusat di sekitar angka nol dan berdistribusi normal, menandakan model bekerja tanpa bias sistemik yang signifikan.